In [11]:
import pandas as pd
import io

# ---------------------------------------------------------
# 1. Load 2024 Data (From Uploaded Images)
# ---------------------------------------------------------
csv_data = """Lead Source,Initiative,# MQLs,MQL Follow Up CVR (S2),Persistence before Max Attempts
Request Contact,NA,979,16%,14.0
Trial,NA,758,6%,14.7
Content,NA,222,1%,5.5
Digital Event,NA,844,0%,6.0
Physical Event,NA,879,1%,6.1
Grand Total,NA,3682,6%,10.4
Request Contact,APAC,761,12%,6.7
Trial,APAC,751,5%,6.1
Content,APAC,102,3%,3.7
Digital Event,APAC,672,1%,2.5
Physical Event,APAC,1815,1%,2.7
Grand Total,APAC,4101,4%,4.4
Request Contact,EMEA,1010,12%,5.6
Trial,EMEA,855,5%,4.7
Content,EMEA,288,1%,3.1
Digital Event,EMEA,361,1%,2.5
Physical Event,EMEA,1900,1%,2.2
Grand Total,EMEA,4414,4%,3.5
Request Contact,LATAM,577,11%,6.3
Trial,LATAM,613,4%,4.8
Content,LATAM,177,1%,4.7
Digital Event,LATAM,792,0%,3.3
Physical Event,LATAM,208,2%,6.2
Grand Total,LATAM,2367,4%,5.0
Request Contact,Digital,1295,8%,4.3
Trial,Digital,2509,2%,2.9
Content,Digital,15,0%,5.3
Digital Event,Digital,49,4%,8.2
Physical Event,Digital,12,0%,3.7
Grand Total,Digital,3880,4%,3.3
Request Contact,SMB,517,14%,7.1
Trial,SMB,679,5%,4.9
Content,SMB,66,3%,4.9
Digital Event,SMB,140,0%,4.4
Physical Event,SMB,62,2%,3.5
Grand Total,SMB,1464,8%,5.5"""

df = pd.read_csv(io.StringIO(csv_data))

# Clean percentages
df['MQL Follow Up CVR (S2)'] = df['MQL Follow Up CVR (S2)'].str.rstrip('%').astype(float) / 100

In [12]:
df['Initiative'] = df['Initiative'].fillna('NA')

In [13]:
# 2. Logic: Value Fetching & Persistence Calculation
# ---------------------------------------------------------

def get_value(initiative, lead_source, col):
    """Fetches a specific value (MQLs or CVR) for a source/initiative."""
    mask = (df['Initiative'] == initiative) & (df['Lead Source'] == lead_source)
    if not mask.any():
        return 0
    return df.loc[mask, col].values[0]

def get_weighted_persistence(initiative, core_status):
    """Calculates weighted average persistence for Core vs Non-Core."""
    # Define groups
    if core_status == 'Core':
        sources = ['Request Contact', 'Trial']
    else:
        sources = ['Content', 'Digital Event','Physical Event']

    # Filter data
    subset = df[(df['Initiative'] == initiative) & (df['Lead Source'].isin(sources))].copy()

    if subset.empty or subset['# MQLs'].sum() == 0:
        return 0

    # Weighted Calc: Sum(Persist * MQLs) / Sum(MQLs)
    weighted_sum = (subset['Persistence before Max Attempts'] * subset['# MQLs']).sum()
    total_mqls = subset['# MQLs'].sum()

    return weighted_sum / total_mqls

# ---------------------------------------------------------
# 3. Build Report (Matching 'Persistence - Sheet1.csv')
# ---------------------------------------------------------

# Column Structure: (Initiative Header, Quarter, Segment, Data_Key_In_DF)
column_map = [
    ('APAC',   '2024Q4', 'Ent/Comm (Std)', 'APAC'),
    ('EMEA',   '2024Q4', 'Ent/Comm (Std)', 'EMEA'),
    ('Global', '2024Q4', 'Digital',        'Digital'),
    ('Global', '2024Q4', 'SMB',            'SMB'),
    ('LATAM',  '2024Q4', 'Ent/Comm (Std)', 'LATAM'),
    ('NA',     '2024Q4', 'Ent/Comm (Std)', 'NA')
]

report_data = {}

for col_def in column_map:
    col_name = (col_def[0], col_def[1], col_def[2]) # MultiIndex Key
    data_initiative = col_def[3]                    # Lookup Key

    col_values = []

    # 1. Conversion Rates (Formatted as %)
    col_values.append(f"{get_value(data_initiative, 'Content', 'MQL Follow Up CVR (S2)'):.0%}")
    col_values.append(f"{get_value(data_initiative, 'Digital Event', 'MQL Follow Up CVR (S2)'):.0%}")
    col_values.append(f"{get_value(data_initiative, 'Physical Event', 'MQL Follow Up CVR (S2)'):.0%}")
    col_values.append(f"{get_value(data_initiative, 'Request Contact', 'MQL Follow Up CVR (S2)'):.0%}")
    col_values.append(f"{get_value(data_initiative, 'Trial', 'MQL Follow Up CVR (S2)'):.0%}")

    # 2. Volumes (Integers)
    col_values.append(int(get_value(data_initiative, 'Grand Total', '# MQLs')))
    col_values.append(int(get_value(data_initiative, 'Request Contact', '# MQLs')))
    col_values.append(int(get_value(data_initiative, 'Content', '# MQLs')))

    # Events Sum (Digital + Physical)
    events_vol = int(get_value(data_initiative, 'Digital Event', '# MQLs') + get_value(data_initiative, 'Physical Event', '# MQLs'))
    col_values.append(events_vol)

    col_values.append(int(get_value(data_initiative, 'Trial', '# MQLs')))

    # 3. Persistence (Weighted Avg)
    core_p = get_weighted_persistence(data_initiative, 'Core')
    non_core_p = get_weighted_persistence(data_initiative, 'Non Core')

    col_values.append(round(core_p, 1))
    col_values.append(round(non_core_p, 1))

    report_data[col_name] = col_values

# ---------------------------------------------------------
# 4. Format & Export
# ---------------------------------------------------------
metrics_rows = [
    "Marketing: Conversion Rate ( Content > S2 )",
    "Marketing: Conversion Rate ( Digital Event > S2 )",
    "Marketing: Conversion Rate ( Physical Event > S2 )",
    "Marketing: Conversion Rate ( Request > S2 )",
    "Marketing: Conversion Rate ( Trial > S2 )",
    "Marketing: MQL Volume",
    "Marketing: MQL volume ( Contact )",
    "Marketing: MQL volume ( Content )",
    "Marketing: MQL volume ( Events )",
    "Marketing: MQL volume ( Trials )",
    "Marketing: Persistence ( Core )",
    "Marketing: Persistence ( Non-Core )"
]

# Create DataFrames with MultiIndex
columns = pd.MultiIndex.from_tuples(report_data.keys(), names=["Initiative", "Quarter", "Segment"])
final_report = pd.DataFrame(report_data.values(), index=columns, columns=metrics_rows).T

# Display & Save
print(final_report.to_markdown())
final_report.to_csv("marketing_persistence_report_2024.csv")

|                                                    | ('APAC', '2024Q4', 'Ent/Comm (Std)')   | ('EMEA', '2024Q4', 'Ent/Comm (Std)')   | ('Global', '2024Q4', 'Digital')   | ('Global', '2024Q4', 'SMB')   | ('LATAM', '2024Q4', 'Ent/Comm (Std)')   | ('NA', '2024Q4', 'Ent/Comm (Std)')   |
|:---------------------------------------------------|:---------------------------------------|:---------------------------------------|:----------------------------------|:------------------------------|:----------------------------------------|:-------------------------------------|
| Marketing: Conversion Rate ( Content > S2 )        | 3%                                     | 1%                                     | 0%                                | 3%                            | 1%                                      | 1%                                   |
| Marketing: Conversion Rate ( Digital Event > S2 )  | 1%                                     | 1%                                     | 4%   